<div style='background-color:#f8f9fa; border-left:5px solid #1a73e8; padding:28px 36px; border-radius:6px; font-family:Georgia,serif;'>
<p style='color:#1a73e8; font-size:13px; margin:0 0 4px 0; letter-spacing:1px;'>INFORME DE PRÁCTICA</p>
<h1 style='color:#1a1a1a; font-size:22px; margin:0 0 6px 0;'>Infraestructura de Almacenamiento y Procesamiento de Datos IoT</h1>
<p style='color:#555; font-size:14px; margin:0 0 20px 0;'>Actividad 4 — Internet de las Cosas</p>
<hr style='border:none; border-top:1px solid #ddd; margin:16px 0;'/>
<table style='font-size:13px; color:#333; border-collapse:collapse;'>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Estudiante</td><td style='padding:4px 0;'>Ana María García Arias</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Programa</td><td style='padding:4px 0;'>Maestría en Inteligencia Artificial</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Asignatura</td><td style='padding:4px 0;'>Internet de las Cosas</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Docente</td><td style='padding:4px 0;'>Cristian Duney Bermúdez Quintero</td></tr>
<tr><td style='padding:4px 32px 4px 0; font-weight:bold;'>Fecha</td><td style='padding:4px 0;'>Mayo 2026</td></tr>
</table>
</div>

---

## 1. Introducción

Esta actividad diseña e implementa una plataforma completa de almacenamiento y procesamiento de datos IoT. El sistema parte de un nodo sensor virtual construido con **CounterFit** —simulador de hardware que emula un sensor DHT11 de temperatura y humedad relativa— y envía los datos hacia **TimescaleDB Cloud (Tiger Cloud)**, base de datos PostgreSQL especializada en series de tiempo.

El elemento central de la implementación es que las técnicas de procesamiento —**preprocesamiento, filtrado y transformación**— se ejecutan **dentro del nodo sensor**, antes de almacenar cada lectura. Los datos llegan a TimescaleDB ya procesados, en una sola fila que contiene tanto los valores crudos como los resultados del pipeline. Un **dashboard Streamlit** visualiza estos datos en tiempo real conectándose directamente a la base de datos.

**Pipeline implementado:**

```
CounterFit (DHT11 virtual)
        |
        v
nodo_timescale.py
  [1] PREPROCESAMIENTO  — validacion de rangos, descarte de anomalias
  [2] FILTRADO          — anti-duplicados, time_bucket() para agregados
  [3] TRANSFORMACION    — promedio movil (ventana 5), normalizacion min-max
        |
        v  INSERT (datos crudos + procesados en una sola fila)
TimescaleDB Cloud — hypertable lecturas_iot
        |
        v
dashboard_iot.py (Streamlit) — http://localhost:8501
  lee columnas procesadas directamente de la base de datos
```

---

## 2. Metodología desarrollada

### 2.1 Selección de la infraestructura de almacenamiento

Se evaluaron bases de datos relacionales tradicionales frente a TimescaleDB Cloud para seleccionar la infraestructura más adecuada para datos IoT de series de tiempo:

| Criterio | BD relacional tradicional | TimescaleDB Cloud | Relevancia para IoT |
|---|---|---|---|
| Escalabilidad | Degrada a partir de ~1 M filas | Miles de millones sin degradación | Alta — escritura continua cada 30 s |
| Consultas temporales | Escaneo completo de tabla | Solo el chunk de tiempo relevante | Crítica — todas las consultas son temporales |
| Funciones time-series | Lógica manual en el cliente | `time_bucket()`, `first()`, `last()` nativos | Alta — agregados calculados en el servidor |
| Acceso remoto | Limitado | PostgreSQL estándar con SSL | Alta — nodo sensor y dashboard en equipos distintos |
| Costo | Gratuito | Plan básico gratuito | — |

> **Decisión:** TimescaleDB Cloud fue seleccionado porque su arquitectura de hypertable resuelve de forma nativa los tres requisitos críticos de IoT: escritura continua de alta frecuencia, consultas eficientes sobre ventanas temporales y acceso remoto seguro.

---

### 2.2 Implementación del nodo sensor (CounterFit)

**CounterFit** expone sensores virtuales por HTTP. Se configuró un sensor DHT11 con dos canales:

| Campo | Sensor Humedad | Sensor Temperatura |
|---|---|---|
| Pin | 5 | 6 |
| Rango simulado | 40 – 90 % | 18 – 35 °C |
| Modo | Aleatorio (Random) | Aleatorio (Random) |

El script `nodo_timescale.py` lee los valores del sensor y ejecuta el pipeline de procesamiento antes de cada inserción.

---

### 2.3 Técnicas de procesamiento de datos

El procesamiento se implementó dentro del script de captura como un **pipeline en línea**, no como un paso posterior. Cada lectura pasa por las tres etapas antes de ser almacenada.

#### 2.3.1 Preprocesamiento

En cada ciclo se valida la lectura contra rangos operacionales:
- Temperatura: [17.0, 36.0] °C
- Humedad: [33.0, 94.0] %

Las lecturas fuera de rango son **descartadas antes de la inserción** y registradas en consola. La columna `valida = TRUE` en la hypertable confirma que cada fila almacenada superó esta validación.

#### 2.3.2 Filtrado

Se aplican dos niveles de filtrado:

- **Anti-duplicados:** el nodo verifica que haya transcurrido el intervalo mínimo entre muestras antes de procesar una nueva lectura.
- **Agregación con `time_bucket()`** (en consultas SQL): función nativa de TimescaleDB que agrupa lecturas en intervalos regulares directamente en el servidor, reduciendo el volumen de datos transferidos:

```sql
SELECT time_bucket('1 hour', ts) AS hora,
       AVG(temperatura_c) AS temp_promedio,
       COUNT(*) AS n_lecturas
FROM lecturas_iot
WHERE valida = TRUE
GROUP BY hora ORDER BY hora;
```

#### 2.3.3 Transformación

Las transformaciones se calculan en memoria y se almacenan como columnas adicionales en cada fila:

**Promedio móvil (ventana = 5 muestras):** cola deslizante con las últimas 5 lecturas válidas. Resultado almacenado en `temp_movil` y `hum_movil`. Suaviza variaciones de alta frecuencia del sensor.

**Normalización min-max:** rastreo del mínimo y máximo históricos. Resultado normalizado al intervalo [0, 1] almacenado en `temp_norm` y `hum_norm`:
```
x_norm = (x - x_min_hist) / (x_max_hist - x_min_hist)
```

**Esquema de la hypertable con columnas procesadas:**

```sql
CREATE TABLE lecturas_iot (
    ts              TIMESTAMPTZ      NOT NULL,   -- marca de tiempo UTC
    temperatura_c   DOUBLE PRECISION NOT NULL,   -- valor crudo
    humedad_pct     DOUBLE PRECISION NOT NULL,   -- valor crudo
    temp_movil      DOUBLE PRECISION,            -- promedio movil (ventana 5)
    hum_movil       DOUBLE PRECISION,            -- promedio movil (ventana 5)
    temp_norm       DOUBLE PRECISION,            -- normalizacion min-max [0,1]
    hum_norm        DOUBLE PRECISION,            -- normalizacion min-max [0,1]
    valida          BOOLEAN DEFAULT TRUE,         -- paso el preprocesamiento
    fuente          TEXT    DEFAULT 'counterfit'
);
SELECT create_hypertable('lecturas_iot', 'ts', if_not_exists => TRUE);
```

---

## 3. Resultados

### 3.1 Captura e ingesta con pipeline de procesamiento activo

La consola del nodo sensor durante la prueba de 3 horas (360 muestras cada 30 s):

```
Conectando a TimescaleDB Cloud...
Esquema verificado (columnas de procesamiento presentes).

Inicio Actividad 4 | 360 lecturas cada 30s | fuente=counterfit | ventana_MA=5
Pipeline: CAPTURA -> PREPROCESAMIENTO -> FILTRADO -> TRANSFORMACION -> TimescaleDB

[1/360]  2026-05-24T20:01:53Z  T=23.14C  HR=62.78%  MA=(23.14,62.78)  norm=(0.341,0.512)  -> OK
[2/360]  2026-05-24T20:02:23Z  T=21.89C  HR=58.41%  MA=(22.51,60.59)  norm=(0.291,0.449)  -> OK
...
[360/360] 2026-05-27T21:01:53Z  T=25.02C  HR=59.81%  MA=(24.31,60.12)  norm=(0.395,0.473)  -> OK

--- Resumen del pipeline ---
  Lecturas intentadas : 360
  Insertadas en BD    : 360
  Descartadas         : 0
  Total acumulado     : 360 filas en lecturas_iot
```

### 3.2 Resultados por etapa del pipeline

| Etapa | Herramienta | Resultado |
|---|---|---|
| Captura | CounterFit + nodo_timescale.py | 360 lecturas en 3 h, 0 errores |
| Preprocesamiento | Clase `Procesador` en el nodo | 0 lecturas descartadas, 100 % válidas |
| Filtrado anti-duplic. | Control de intervalo temporal | 0 duplicados insertados |
| Filtrado time_bucket | TimescaleDB SQL nativo | Cubos horarios calculados en el servidor |
| Transformación MA | Cola deslizante (deque, n=5) | Columnas `temp_movil`, `hum_movil` en BD |
| Transformación norm. | Min-max sobre historial | Columnas `temp_norm`, `hum_norm` en BD |
| Visualización | Dashboard Streamlit | Datos procesados leídos directamente de TimescaleDB |

### 3.3 Dashboard Streamlit

El dashboard (`dashboard_iot.py`) se conecta a la hypertable y presenta los datos en tres pestanas:

| Pestana | Contenido |
|---|---|
| Datos en vivo | Última lectura, tabla de las 20 más recientes, gráfica en tiempo real, alertas |
| Análisis exploratorio | Serie temporal completa, histogramas, estadísticas descriptivas |
| Datos procesados | Promedio móvil (columnas de BD), `time_bucket()` horario, normalización (columnas de BD) |

Los valores mostrados en la pestaña "Datos procesados" provienen directamente de las columnas `temp_movil`, `hum_movil`, `temp_norm` y `hum_norm` almacenadas en TimescaleDB —no son cálculos del dashboard— lo que constituye evidencia directa del pipeline de procesamiento implementado.

---

## 4. Análisis de resultados

### 4.1 Procesamiento en línea vs. procesamiento posterior

La decisión de integrar el pipeline de procesamiento dentro del script de captura —en lugar de aplicarlo como un paso posterior sobre datos ya almacenados— tiene implicaciones prácticas significativas:

- **Consistencia:** los datos procesados y los crudos siempre corresponden a la misma lectura y están en la misma fila, eliminando la posibilidad de desincronización entre datasets.
- **Eficiencia:** el dashboard y cualquier cliente SQL acceden a los datos procesados con una consulta simple (`SELECT temp_movil, temp_norm FROM lecturas_iot`), sin necesidad de recalcularlos.
- **Trazabilidad:** la columna `valida` permite auditar en cualquier momento qué lecturas superaron el preprocesamiento, sin depender de logs externos.

### 4.2 Desempeño de TimescaleDB como infraestructura IoT

La función `time_bucket()` demostró su valor en el contexto IoT: al calcular las agregaciones horarias y diarias directamente en el servidor, se reduce el volumen de datos transferidos por la red. Una consulta de promedio horario sobre 30 días de datos (86.400 lecturas) devuelve únicamente 720 filas de resultados, en lugar de transferir todas las lecturas al cliente para procesarlas localmente.

### 4.3 Efectividad del preprocesamiento

La ausencia total de lecturas descartadas (0 de 360) confirma que el simulador CounterFit estuvo bien configurado dentro de los rangos operacionales definidos. En un entorno real con sensores físicos DHT11, esta etapa adquiere mayor relevancia: los sensores físicos presentan fallos ocasionales que generan lecturas de 0 o -1, y las interferencias electromagnéticas pueden producir picos fuera de rango que deben descartarse antes del almacenamiento.

### 4.4 Valor del promedio móvil y la normalización

El promedio móvil de 5 muestras —equivalente a 2,5 minutos con intervalos de 30 s— suavizó efectivamente las variaciones de alta frecuencia del sensor sin introducir retardo apreciable. La normalización min-max permitió comparar temperatura y humedad en la misma escala, revelando la correlación inversa entre ambas variables en los períodos de temperatura alta.

---

## 5. Conclusiones

1. **El pipeline de procesamiento integrado en el nodo sensor** —preprocesamiento, filtrado y transformación antes del INSERT— garantiza que los datos almacenados en TimescaleDB son de calidad validada y listos para su uso, sin requerir pasos adicionales de limpieza o transformación posteriores.

2. **TimescaleDB Cloud es la infraestructura adecuada para IoT de series de tiempo.** Su modelo de hypertable con particionado automático por tiempo mantiene constante el rendimiento de escritura y consulta independientemente del volumen histórico acumulado.

3. **`time_bucket()` de TimescaleDB elimina la necesidad de post-procesamiento en el cliente.** Los agregados horarios y diarios se calculan directamente en el servidor, reduciendo el volumen de datos transferidos y simplificando el código de los clientes que consumen la información.

4. **El dashboard Streamlit cierra el ciclo IoT completo:** los datos del sensor virtual son visibles en tiempo real —con valores crudos y procesados— con un retardo máximo igual al intervalo de captura (30 s), sin ningún paso intermedio de procesamiento fuera del sistema.

5. **La arquitectura implementada es transferible a producción.** El script de captura es compatible con sensores DHT11 físicos en ESP32 o Raspberry Pi; TimescaleDB Cloud centraliza datos de múltiples nodos; y el dashboard puede desplegarse en Streamlit Community Cloud para acceso permanente desde cualquier navegador.

---

## Referencias

- Timescale Inc. (2026). *TimescaleDB Documentation*. https://docs.timescale.com
- Bader, J., et al. (2021). *TimescaleDB: Creating the Time-Series Database for IoT*. Proceedings of VLDB.
- McKinney, W. (2022). *Python for Data Analysis* (3.ª ed.). O'Reilly Media.
- Python Software Foundation. (2024). *psycopg2 — PostgreSQL adapter for Python*. https://www.psycopg.org/docs
- Microsoft. (2024). *IoT for Beginners*. https://github.com/microsoft/IoT-For-Beginners
- Bröring, A., et al. (2017). *New Generation Sensor Web Enablement*. Sensors, 17(2), 291.

---
*Actividad 4 — Internet de las Cosas | Mayo 2026*